# One building — the simulator

One building, three lift units, five minutes of a morning. This notebook puts the
events on Kafka; `2_building_monitoring.ipynb` reads them back with Spark.

**Open both, side by side, with this one on the LEFT and the monitoring notebook
on the RIGHT.** You will move between them six times, and the arrows do not mean
anything until the two windows are in that order:

* a heading with **❯❯** says *go to the notebook on the right*;
* a heading with **❮❮**, in the other notebook, says *come back to this one on
  the left*.

Each of them also says what to run once you are there, and where to pick up when
you return — which is always the cell **below** the arrow, never one above it.

---
## How to run the two notebooks

They run **interleaved**, and the order matters because the point of the lecture
is to watch the answers change while the morning happens.

1. **here**, from the top down to and including the cell that creates the
   producer and defines `send(...)` — the last one before *1) The morning*. On
   the way it prints `37 events` and the five per-minute counts. Nothing has been
   sent yet, and `send` now exists, which the next steps need;
2. **`2_building_monitoring.ipynb`**, from the top down to the first query
   `Q.9.1` and its `SELECT`. It will show an empty table, which is correct;
3. **back here**, *1) The morning, one minute at a time*;
4. **back to the monitoring notebook**, down to and including `Q.9.5`, the
   self-join. Its table will be empty too, and that is also correct;
5. **back here**, *2) One floor, four openings*, and then the cell that moves the
   clock on;
6. **back to the monitoring notebook**, and read `Q.9.5` — re-running the
   `SELECT` cells, here as everywhere, until the batch has landed;
7. at the end: stop the queries **in the monitoring notebook**, then run
   *Clean up* here.

You can also send everything first and only then start Spark —
`startingOffsets` is `earliest`, so nothing is lost either way. It is a fine way
to check that the stack works, and a poor way to teach: with every event already
on the topic Spark reads them in one batch, and two of the things this notebook
is built to show — how `update` mode reports a window, and how `append` mode
holds a joined row back — never happen in front of anybody.

### The client is already installed
The container installs `confluent-kafka` before Jupyter starts, so this is a
check and not a wait.

In [1]:
import confluent_kafka
print("confluent-kafka", confluent_kafka.version()[0])

confluent-kafka 2.5.0


In [2]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer
import json, time

bootstrap_server = "kafka:9092"
TOPIC = "building-events"

admin = AdminClient({"bootstrap.servers": bootstrap_server})

## The topic, and why it has three partitions

One partition per lift unit, and the key of every message is the `unitId`. That
is the decision of lecture 8 applied to this building: a key's partition is
`hash(key) % n`, so all the events of one unit land in one partition and stay in
order with respect to each other.

**Predict:** three units and three partitions — does each unit get a partition of
its own?

In [3]:
for t, f in admin.create_topics(
        [NewTopic(TOPIC, num_partitions=3, replication_factor=1)]).items():
    try:
        f.result()
        print("created", t)
    except Exception as e:
        print("topic", t, "->", e)

created building-events


In [4]:
md_ = admin.list_topics(timeout=10)
for name, t in md_.topics.items():
    if name == TOPIC:
        print(name, "->", len(t.partitions), "partitions")

building-events -> 3 partitions


## The building

Three units in one building, in two banks. `EU-001` and `EU-002` serve the lower
floors, `EU-003` serves the upper ones — which is why it is still asleep in the
first minute and busiest two minutes later, when the people who came through the
lobby reach it.

The trace is **designed, not random**: every number below is a decision, and the
monitoring notebook's answers can be checked against it by hand.

Event time lives in the payload, so the producer can send as fast as it likes.
Nothing here depends on wall-clock speed.

In [5]:
# -*- coding: utf-8 -*-
"""Five minutes of one building's doors, as a function of two decisions:
which bank a unit belongs to, and what the traffic does to that bank."""
from datetime import datetime, timedelta

T0 = datetime(2026, 10, 14, 8, 0, 0)        # building time zero, UTC

# unit -> bank. The monitoring notebook joins the stream against this.
BANK = {"EU-001": "low", "EU-002": "low", "EU-003": "high"}

# doors per unit per minute, minute 0..4. The lobby moves first, the high-rise
# bank a minute later: that lag is the point of the stream-to-table join.
RATE = {
    "EU-001": [2, 3, 5, 3, 2],
    "EU-002": [1, 2, 4, 2, 1],
    "EU-003": [0, 1, 6, 4, 1],
}

FLOORS = 10


def ev(kind, unit, ts, **kw):
    d = {"event": kind, "unitId": unit, "ts": ts.isoformat(timespec="seconds")}
    d.update(kw)
    return d


def morning():
    """37 door events over five minutes, spread inside each minute
    deterministically so that two runs produce the same trace."""
    out = []
    for unit, rates in RATE.items():
        seed = int(unit.split("-")[1])
        for minute, rate in enumerate(rates):
            for k in range(rate):
                sec = (seed * 7 + k * 17) % 60
                ts = T0 + timedelta(minutes=minute, seconds=sec)
                car = "A" if (seed + k) % 2 == 0 else "B"
                floor = (seed * 3 + k * 5) % FLOORS
                out.append(ev("DoorOpened", unit, ts, car=car, floor=floor,
                              servedDir="UP" if (seed + k) % 2 == 0 else "DOWN"))
    return sorted(out, key=lambda e: e["ts"])


def bunching():
    """Four openings at one unit, one floor, one direction, inside twenty
    seconds. Two cars, alternating: A, B, A, B at +0, +8, +14, +18.

    It reads as two bunchings, and that is the trap. The four events are closer
    together than the rule's own window, so they are not two of anything: the
    join will pair the first opening with the *last* one as well, eighteen
    seconds apart, and the followed-by pattern of lecture 7 will not."""
    u, base = "EU-003", T0 + timedelta(minutes=7)
    return [ev("DoorOpened", u, base + timedelta(seconds=s),
               car=c, floor=3, servedDir="UP")
            for s, c in ((0, "A"), (8, "B"), (14, "A"), (18, "B"))]


def tail():
    """Two ordinary openings two minutes later, at another floor, so that they
    join with nothing.

    Their only job is to carry the event time forward. In append mode a joined
    row is withheld until the watermark says that no partner can still arrive,
    so without something later than the bunching the result never becomes
    final and the query prints an empty table for ever."""
    u, base = "EU-001", T0 + timedelta(minutes=9)
    return [ev("DoorOpened", u, base + timedelta(seconds=s),
               car=c, floor=0, servedDir=d)
            for s, c, d in ((0, "A", "UP"), (30, "B", "DOWN"))]


In [6]:
rows = morning()
print(len(rows), "events")
for m in range(5):
    n = len([r for r in rows if r["ts"][14:16] == f"{m:02d}"])
    print(f"  minute {m}: {n}")

37 events
  minute 0: 3
  minute 1: 6
  minute 2: 15
  minute 3: 9
  minute 4: 4


Thirty-seven events, and the shape of the morning is already visible in those
five numbers. Write them down: the monitoring notebook has to agree with them.

In [7]:
# the plain Producer, not SerializingProducer: the library marks that one
# "experimental and likely to be removed", and produce() already encodes a
# str key or value as utf-8 by itself
producer = Producer({"bootstrap.servers": bootstrap_server})

def send(events):
    """key = unitId, deliberately: it is the partitioning key of the lecture."""
    for r in events:
        producer.produce(topic=TOPIC, key=r["unitId"], value=json.dumps(r))
    producer.flush()
    print(f"sent {len(events)} events")

---
# ❯❯ now the notebook on the right

In `2_building_monitoring.ipynb`: run it from the top down to the `SELECT` of
`Q.9.1`, which will print an **empty table** — that is what you are going there
to see.

Then come back here, to the cell below this one.

## 1) The morning, one minute at a time

Sent minute by minute, with a pause between them, so that the monitoring
notebook can be watched while it fills rather than after it has finished.

In [8]:
for m in range(5):
    send([r for r in rows if r["ts"][14:16] == f"{m:02d}"])
    time.sleep(2)

sent 3 events
sent 6 events
sent 15 events
sent 9 events
sent 4 events


---
# ❯❯ now the notebook on the right

Five minutes are on the topic. In `2_building_monitoring.ipynb`: carry on
downwards, from `Q.9.2` to the two queries of `Q.9.5` and the two `SELECT`s under
them — which come back **empty**, and that is correct too.

Then come back here, to the cell below this one.

## 2) One floor, four openings

Everything so far was traffic. This is the configuration the next section of the
monitoring notebook is built on: **one unit, one floor, one direction, four
openings inside eighteen seconds**, two cars alternating.

Run the monitoring notebook as far as `Q.9.5` **before** you send this, so that
the join is already watching.

**Predict**, and write the three numbers down before you look:

* how many rows does a self-join on *same unit, same floor, same direction,
  different car, within twenty seconds* produce?
* how many does it produce once you add that `b` must come **after** `a`?
* and how many rows would lecture 7's `every a -> b` have produced on the same
  four events?

In [9]:
send(bunching())
for e in bunching():
    print(e["ts"][11:], e["car"], "floor", e["floor"], e["servedDir"])

sent 4 events
08:07:00 A floor 3 UP
08:07:08 B floor 3 UP
08:07:14 A floor 3 UP
08:07:18 B floor 3 UP


### and now move the clock on

Two ordinary openings two minutes later, at another floor, so they pair with
nothing. They are here only to carry the event time forward: in `append` mode a
joined row is held back until the watermark says that no partner can still
arrive, so without something later than the bunching the join's table stays
empty and looks broken.

That is the third of the things EPL never had to say — and in EPL
`timer:within(20 sec)` was the business rule *and* the bound on state, while
here they are two separate decisions.

In [10]:
import time
time.sleep(2)
send(tail())

sent 2 events


---
# ❯❯ now the notebook on the right

Everything is on the topic. In `2_building_monitoring.ipynb`: read the rest of
it, straight down — the two `SELECT`s of `Q.9.5` again, then section 8 — and run
its *Clean up*, which stops the queries.

Then come back here **one last time**, for the cell below.

## Clean up

Run this at the end of the lecture, **after** stopping the queries in the other
notebook. Deleting the topic under a running query is a good way to see an
exception you did not mean to produce.

In [11]:
for t, f in admin.delete_topics([TOPIC], operation_timeout=30).items():
    try:
        f.result()
        print("deleted", t)
    except Exception as e:
        print(t, "->", e)

deleted building-events
